In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs and parse timestamps as UTC datetimes
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Aggregate metric.csv by cmdb_id and kpi_name
metric_agg = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        min_value='min',
        median_value=lambda x: x.median(),
        p95_value=lambda x: x.quantile(0.95),
        max_value='max'
    )
    .reset_index()
)

metric_agg_top = metric_agg.sort_values('count', ascending=False).head(50)

# Distinct combinations for log.csv and trace.csv (limit 50)
log_distinct = log[['cmdb_id', 'log_name']].drop_duplicates().reset_index(drop=True).head(50)
trace_distinct = trace[['cmdb_id', 'trace_name']].drop_duplicates().reset_index(drop=True).head(50)

# error_logs: count rows per cmdb_id (limit 20)
error_counts = (
    error_logs
    .groupby('cmdb_id')
    .size()
    .reset_index(name='error_count')
    .sort_values('error_count', ascending=False)
    .head(20)
)

# Display the compact results (IPython will show these variables)
metric_agg_top, log_distinct, trace_distinct, error_counts

```
Out[1]:
```
```python
# Summarize the previously computed results in plain English (display as a single string).
metric_groups = metric_agg_top.shape[0]
typical_count = int(metric_agg_top['count'].mode()[0]) if not metric_agg_top.empty else 0
sample_metrics = metric_agg_top.head(3).apply(
    lambda r: f"{r['cmdb_id']}:{r['kpi_name']} (count={int(r['count'])}, median={r['median_value']:.3g}, max={r['max_value']:.3g})",
    axis=1
).tolist()

log_pairs_count = log_distinct.shape[0]
trace_pairs_count = trace_distinct.shape[0]
error_cmdb_count = error_counts.shape[0]
total_error_rows = int(error_logs.shape[0]) if 'error_logs' in globals() else 0

summary = (
    f"Summary of telemetry processing:\n"
    f"- Metrics: {metric_groups} aggregated (top groups shown). Most groups have count ≈ {typical_count}.\n"
    f"  Example metric groups: {', '.join(sample_metrics)}.\n"
    f"- Logs: {log_pairs_count} distinct (cmdb_id, log_name) combinations found (examples include carts, catalogue, front-end, orders, payment, etc.).\n"
    f"- Traces: {trace_pairs_count} distinct (cmdb_id, trace_name) combinations found (none present in this dataset).\n"
    f"- Error logs: {total_error_rows} total rows; {error_cmdb_count} cmdb_id(s) with error counts (none present in the results).\n"
    f"Conclusion: Metrics and log name combinations are present and aggregated; there are no trace entries and no error-log rows in the provided files."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count     min_value  median_value     p95_value     max_value
0          carts         cpu     25  1.296685e+00  1.465656e+00  1.845081e+00  3.574640e+00
1          carts  latency-50     25  1.788330e-02  1.882167e-02  1.984109e-02  1.988057e-02
2          carts  latency-90     25  4.018466e-02  4.295985e-02  4.638586e-02  4.816010e-02
3          carts         mem     25  2.069184e+08  2.071902e+08  2.072533e+08  2.080969e+08
4          carts      socket     25  1.038333e+01  1.100000e+01  1.229000e+01  1.273333e+01
..           ...         ...    ...           ...           ...           ...           ...
45  queue-master      diskio     25  2.385432e+04  4.083794e+04  5.352855e+04  5.581885e+04
46  queue-master         mem     25  2.568493e+08  2.575130e+08  2.579987e+08  2.588383e+08
47  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
48      rabbitmq         cpu     25  6.550629e-01  7.894645e-01  8.309264e-01  8.362924e-01
49      rabbitmq      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 7 columns],          cmdb_id         log_name
0          carts  log.error_count
1          carts    log.row_count
2       carts-db  log.error_count
3       carts-db    log.row_count
4      catalogue  log.error_count
5      catalogue    log.row_count
6      front-end  log.error_count
7      front-end    log.row_count
8         orders  log.error_count
9         orders    log.row_count
10       payment  log.error_count
11       payment    log.row_count
12  queue-master  log.error_count
13  queue-master    log.row_count
14      shipping  log.error_count
15      shipping    log.row_count
16          user  log.error_count
17          user    log.row_count
18    session-db  log.error_count
19    session-db    log.row_count, Empty DataFrame
Columns: [cmdb_id, trace_name]
Index: [], Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing metric if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Parse timestamps as UTC datetimes
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Services of interest
services = ['carts','catalogue','orders','payment','user']

# Restrict consideration to groups whose cmdb_id is in services for global P95 calculation
metric_services = metric[metric['cmdb_id'].isin(services)].copy()

# Rule: compute global P95 using the entire available series for each (cmdb_id, kpi_name) BEFORE filtering by window
global_p95 = (
    metric_services
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Define incident window (UTC)
start = pd.to_datetime('2024-01-19 21:56:19', utc=True)
end   = pd.to_datetime('2024-01-19 22:26:19', utc=True)

# Filter original metric rows to the incident window and to the services list
metric_window = metric_services[(metric_services['timestamp'] >= start) & (metric_services['timestamp'] <= end)].copy()

# If there are no rows in the window at all, return explicit empty indicator
if metric_window.empty:
    output = "EMPTY: no metric rows for specified services in the incident window."
else:
    # For each group present in the window, compute the required summary,
    # joining with the precomputed global_P95 values.
    window_groups = (
        metric_window
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            total_points=('value', 'count'),
            max_value_in_window=('value', 'max'),
        )
        .reset_index()
    )

    # Attach global_P95
    window_groups = window_groups.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

    # Compute anomaly_count and earliest_anomaly_timestamp per group using metric_window rows
    # (value >= global_P95 counts as anomaly)
    # Prepare a helper DF with anomalies flagged (taking care of groups where global_P95 may be NaN)
    mw = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
    # Flag anomalies (if global_P95 is NaN, treat as no anomaly)
    mw['is_anomaly'] = False
    mw.loc[~mw['global_P95'].isna(), 'is_anomaly'] = mw.loc[~mw['global_P95'].isna(), 'value'] >= mw.loc[~mw['global_P95'].isna(), 'global_P95']

    anomaly_agg = (
        mw[mw['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
        .reset_index()
    )

    # Merge anomaly stats into window_groups (fill missing anomaly counts with 0)
    result = window_groups.merge(anomaly_agg, on=['cmdb_id', 'kpi_name'], how='left')
    result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
    # Format earliest_anomaly_timestamp as ISO UTC string or null
    def fmt_ts(x):
        if pd.isna(x):
            return None
        # Ensure UTC ISO without offset (Z)
        return pd.to_datetime(x).strftime('%Y-%m-%dT%H:%M:%SZ')
    result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].apply(fmt_ts)

    # Compute severity_ratio = max_value_in_window / global_P95 (or null if global_P95 == 0 or NaN)
    def compute_severity(row):
        gp = row['global_P95']
        if pd.isna(gp) or gp == 0:
            return np.nan
        return row['max_value_in_window'] / gp

    result['severity_ratio'] = result.apply(compute_severity, axis=1)

    # Select and order columns as requested
    result = result[
        ['cmdb_id', 'kpi_name', 'total_points', 'anomaly_count', 'earliest_anomaly_timestamp',
         'max_value_in_window', 'global_P95', 'severity_ratio']
    ]

    # Sort by anomaly_count desc then severity_ratio desc
    result = result.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

    # Limit to top 20 rows
    result = result.head(20)

    # If no anomalies across all groups, return explicit empty indicator
    if result['anomaly_count'].sum() == 0:
        output = "EMPTY: no anomalies found for specified services in the incident window."
    else:
        output = result

# Display the compact output variable (either a DataFrame or the empty-indicator string)
output

```
Out[4]:
```
```python
# Build a plain-English summary string from the previously computed 'result' DataFrame.
# If 'result' is not available, produce a fallback message.
import pandas as pd
if 'result' not in globals():
    summary = "No result DataFrame available in the session to summarize."
else:
    df = result.copy()
    # Only keep anomalous groups
    df_anom = df[df['anomaly_count'] > 0].copy()
    total_groups = len(df_anom)
    # Top by anomaly_count then severity_ratio (already sorted in previous step, but ensure)
    top_by_count = df_anom.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).head(5)
    # Highest severity_ratio overall
    top_severity = df_anom.sort_values('severity_ratio', ascending=False).head(1)
    # Examples of groups with 2 anomalies (common case)
    two_anom_examples = df_anom[df_anom['anomaly_count'] == 2].head(5)

    lines = []
    lines.append(f"Summary for services [carts, catalogue, orders, payment, user]:")
    lines.append(f"- Total metric groups with >=1 anomaly in the incident window: {total_groups}")
    if total_groups == 0:
        lines.append("- EMPTY: no anomalies found for these services in the window.")
    else:
        lines.append("- Top anomalous groups by anomaly count:")
        for _, r in top_by_count.iterrows():
            lines.append(
                f"  • {r['cmdb_id']}:{r['kpi_name']} — anomaly_count={int(r['anomaly_count'])}, "
                f"total_points={int(r['total_points'])}, earliest_anomaly={r['earliest_anomaly_timestamp']}, "
                f"max_in_window={r['max_value_in_window']:.6g}, global_P95={r['global_P95']:.6g}, "
                f"severity_ratio={r['severity_ratio']:.3g}"
            )
        # Highest severity detail
        ts = top_severity.iloc[0]
        lines.append(
            f"- Highest severity relative to historical P95: {ts['cmdb_id']}:{ts['kpi_name']} "
            f"(severity_ratio={ts['severity_ratio']:.3g}, max={ts['max_value_in_window']:.6g}, P95={ts['global_P95']:.6g}, "
            f"earliest_anomaly={ts['earliest_anomaly_timestamp']})"
        )
        if not two_anom_examples.empty:
            ex = ", ".join([f"{r['cmdb_id']}:{r['kpi_name']}" for _, r in two_anom_examples.iterrows()])
            lines.append(f"- Several groups had 2 anomaly points (examples): {ex}")
        lines.append("- Interpretation:")
        lines.append("  • Socket metrics in payment and user were fully anomalous across the window (25/25 points).")
        lines.append("  • Catalogue socket had a sustained anomaly (10 points).")
        lines.append("  • The most extreme relative spike was catalogue:diskio (severity ≈ 6.75), indicating a large disk I/O spike versus its historical P95.")
        lines.append("  • Other anomalies are smaller (severity ≈ 1.02–1.94) and affect cpu, latency, workload, and mem KPIs across services.")
        lines.append("Conclusion: Investigate socket issues in payment/user and the large disk I/O spike on catalogue first; follow up on CPU/latency/workload anomalies as secondary signals.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_P95  severity_ratio
23    payment      socket            25             25       2024-01-19T21:59:00Z         5.000000e+00  5.000000e+00        1.000000
29       user      socket            25             25       2024-01-19T21:59:00Z         1.700000e+01  1.700000e+01        1.000000
11  catalogue      socket            25             10       2024-01-19T22:14:00Z         1.300000e+01  1.300000e+01        1.000000
7   catalogue      diskio            25              2       2024-01-19T21:59:00Z         1.515048e+05  2.244583e+04        6.749799
0       carts         cpu            25              2       2024-01-19T22:03:00Z         3.574640e+00  1.845081e+00        1.937389
9   catalogue  latency-90            25              2       2024-01-19T22:22:00Z         3.587980e-01  2.841766e-01        1.262588
15     orders  latency-90            25              2       2024-01-19T22:16:00Z         1.423439e-01  1.255206e-01        1.134028
10  catalogue         mem            25              2       2024-01-19T22:21:00Z         5.337263e+08  4.897205e+08        1.089859
18     orders    workload            25              2       2024-01-19T22:02:00Z         2.319900e+00  2.136050e+00        1.086070
13     orders         cpu            25              2       2024-01-19T22:04:00Z         3.008598e+00  2.818771e+00        1.067344
24    payment    workload            25              2       2024-01-19T22:17:00Z         2.373400e+00  2.233990e+00        1.062404
25       user         cpu            25              2       2024-01-19T22:12:00Z         9.208620e-01  8.807338e-01        1.045562
17     orders      socket            25              2       2024-01-19T22:08:00Z         1.665000e+01  1.597333e+01        1.042362
30       user    workload            25              2       2024-01-19T22:01:00Z         2.003340e+01  1.929130e+01        1.038468
2       carts  latency-90            25              2       2024-01-19T22:13:00Z         4.816010e-02  4.638586e-02        1.038250
4       carts      socket            25              2       2024-01-19T22:20:00Z         1.273333e+01  1.229000e+01        1.036073
8   catalogue  latency-50            25              2       2024-01-19T22:14:00Z         4.666551e-03  4.514256e-03        1.033736
27       user  latency-90            25              2       2024-01-19T22:16:00Z         1.261215e-02  1.226866e-02        1.027998
12  catalogue    workload            25              2       2024-01-19T22:01:00Z         4.392367e+00  4.300547e+00        1.021351
19    payment         cpu            25              2       2024-01-19T22:01:00Z         9.766311e-02  9.593511e-02        1.018012```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
from pandas.api import types as ptypes

# Reuse existing 'log' variable if present; otherwise load
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

# Ensure timestamp is parsed as UTC datetimes
if not ptypes.is_datetime64_any_dtype(log['timestamp']):
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
else:
    # If it's datetime but not timezone-aware, convert to UTC
    if log['timestamp'].dt.tz is None:
        log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Services (cmdb_ids) of interest
services = ["payment","user","catalogue","carts","orders"]

# Restrict to those services for global threshold calculation
log_services = log[log['cmdb_id'].isin(services)].copy()

# Compute global P95 per (cmdb_id, log_name) using the entire available series BEFORE windowing
global_p95 = (
    log_services
    .groupby(['cmdb_id', 'log_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Define incident window (UTC)
start = pd.to_datetime('2024-01-19 21:56:19', utc=True)
end   = pd.to_datetime('2024-01-19 22:26:19', utc=True)

# Filter original log rows to the incident window
log_window = log_services[(log_services['timestamp'] >= start) & (log_services['timestamp'] <= end)].copy()

# If no rows in window at all, return explicit empty indicator
if log_window.empty:
    output = "EMPTY: no log rows for specified services in the incident window."
else:
    # Aggregate per-group stats in window
    window_groups = (
        log_window
        .groupby(['cmdb_id', 'log_name'])
        .agg(
            total_points=('value', 'count'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )

    # Attach global_P95
    window_groups = window_groups.merge(global_p95, on=['cmdb_id', 'log_name'], how='left')

    # Flag anomalies per row (value >= global_P95). If global_P95 is NaN, treat as no anomaly.
    lw = log_window.merge(global_p95, on=['cmdb_id', 'log_name'], how='left')
    lw['is_anomaly'] = False
    mask = ~lw['global_P95'].isna()
    lw.loc[mask, 'is_anomaly'] = lw.loc[mask, 'value'] >= lw.loc[mask, 'global_P95']

    # Aggregate anomaly counts and earliest anomaly timestamps
    anomaly_agg = (
        lw[lw['is_anomaly']]
        .groupby(['cmdb_id', 'log_name'])
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
        .reset_index()
    )

    # Merge anomaly stats
    result = window_groups.merge(anomaly_agg, on=['cmdb_id', 'log_name'], how='left')
    result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)

    # Format earliest_anomaly_timestamp as ISO UTC string or None
    def fmt_ts(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(x).strftime('%Y-%m-%dT%H:%M:%SZ')
    result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].apply(fmt_ts)

    # Compute severity_ratio = max_value_in_window / global_P95 (null if global_P95 is NaN or 0)
    def compute_severity(row):
        gp = row['global_P95']
        if pd.isna(gp) or gp == 0:
            return np.nan
        return row['max_value_in_window'] / gp
    result['severity_ratio'] = result.apply(compute_severity, axis=1)

    # Select required columns and sort
    result = result[
        ['cmdb_id', 'log_name', 'total_points', 'anomaly_count', 'earliest_anomaly_timestamp',
         'max_value_in_window', 'global_P95', 'severity_ratio']
    ].sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20)

    # If no anomalies across all groups, return explicit empty indicator
    if result['anomaly_count'].sum() == 0:
        output = "EMPTY: no anomalies found for specified services in the incident window."
    else:
        output = result

# Display the compact result (DataFrame or empty-indicator string)
output

```
Out[5]:
```
```python
# Summarize the log anomaly findings in plain English as a single string.
if 'result' in globals():
    df = result.copy()
    lines = []
    lines.append("Log analysis summary for services [payment, user, catalogue, carts, orders] during 2024-01-19 21:56:19 to 2024-01-19 22:26:19 UTC:")
    lines.append("")
    # log.error_count observations
    lec = df[df['log_name'] == 'log.error_count']
    if not lec.empty:
        lines.append("- log.error_count:")
        lines.append("  • All services show anomaly_count equal to total_points (23–25) with earliest anomaly at 2024-01-19T21:59:00Z.")
        lines.append("  • max_value_in_window = 0 and global_P95 = 0 for these groups, so the anomaly flag arises from value >= P95 (0 >= 0).")
        lines.append("  • Interpretation: this is NOT a meaningful error spike (values are zero); severity is undefined because P95==0. Treat as non-actionable unless other evidence shows real errors.")
        lines.append("")
    # log.row_count observations
    lrc = df[df['log_name'] == 'log.row_count'].copy()
    if not lrc.empty:
        lines.append("- log.row_count (genuine spikes):")
        for _, r in lrc.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).iterrows():
            lines.append(
                f"  • {r['cmdb_id']}: total_points={int(r['total_points'])}, anomaly_count={int(r['anomaly_count'])}, "
                f"earliest_anomaly={r['earliest_anomaly_timestamp']}, max={int(r['max_value_in_window'])}, "
                f"P95={r['global_P95']:.1f}, severity_ratio={r['severity_ratio']:.3f}"
            )
        lines.append("")
        lines.append("  • Notable items:")
        lines.append("    - orders and payment each had 6 anomalous points (earliest ~2024-01-19T22:01:00Z) with modest increases over P95 (orders ≈ +3%, payment ≈ +1.9%).")
        lines.append("    - carts and catalogue show smaller or exact-P95 spikes; user shows a small rise (~+0.8%).")
        lines.append("  • Interpretation: these are modest increases in log row counts (throughput or logging rate). Investigate increased activity around ~22:01 UTC for orders/payment/user and ~22:09 for catalogue.")
        lines.append("")
    lines.append("Overall recommendation:")
    lines.append("- Prioritize investigating increased log.row_count for orders and payment (highest anomaly counts and relative increase).")
    lines.append("- Do NOT treat the log.error_count 'anomalies' as real errors because values are zero and P95 is zero; they are artifacts of the >=P95 rule when P95==0.")
    summary = "\n".join(lines)
else:
    summary = "No result DataFrame available to summarize."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  total_points  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_P95  severity_ratio
2  catalogue  log.error_count            25             25       2024-01-19T21:59:00Z                    0         0.0             NaN
4     orders  log.error_count            25             25       2024-01-19T21:59:00Z                    0         0.0             NaN
6    payment  log.error_count            25             25       2024-01-19T21:59:00Z                    0         0.0             NaN
8       user  log.error_count            25             25       2024-01-19T21:59:00Z                    0         0.0             NaN
0      carts  log.error_count            23             23       2024-01-19T21:59:00Z                    0         0.0             NaN
5     orders    log.row_count            25              6       2024-01-19T22:01:00Z                  134       130.0        1.030769
7    payment    log.row_count            25              6       2024-01-19T22:01:00Z                  107       105.0        1.019048
1      carts    log.row_count            23              5       2024-01-19T22:02:00Z                  100       100.0        1.000000
3  catalogue    log.row_count            25              3       2024-01-19T22:09:00Z                  170       170.0        1.000000
9       user    log.row_count            25              2       2024-01-19T22:01:00Z                  623       617.8        1.008417```
```